In [ ]:
# Install all required libraries
!pip install -q google-play-scraper pandas numpy matplotlib seaborn wordcloud vaderSentiment spacy scikit-learn langdetect

# Download spaCy English model
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import time

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Natural Language Processing
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, LangDetectException

# Google Play Store Scraper
from google_play_scraper import reviews, Sort


In [ ]:
# Dictionary of app names and their corresponding Google Play Store IDs
apps = {
    'CBE': 'com.combanketh.mobilebanking',
    'BOA': 'com.boa.boaMobileBanking',
    'Dashen': 'com.dashen.dashensuperapp'
}


In [ ]:
def scrape_reviews(apps, max_reviews=1000):
    """
    Scrape up to `max_reviews` reviews for each app using google-play-scraper.
    Applies batch handling, delay, and error fallback.
    Returns a combined DataFrame of all scraped reviews.
    """
    all_reviews = []

    for bank, app_id in apps.items():
        print(f"\nScraping reviews for {bank}...")
        result = []
        batch_count = 0
        MAX_BATCHES = 10
        BATCH_SIZE = 200

        try:
            partial, continuation_token = reviews(
                app_id,
                lang='en',
                country='et',
                sort=Sort.NEWEST,
                count=BATCH_SIZE
            )
            result.extend(partial)
            batch_count += 1
            print(f"Batch {batch_count}: Fetched {len(result)} reviews.")

            while len(result) < max_reviews and continuation_token and batch_count < MAX_BATCHES:
                more, continuation_token = reviews(
                    app_id,
                    continuation_token=continuation_token,
                    lang='en',
                    country='et'
                )
                result.extend(more)
                batch_count += 1
                print(f"Batch {batch_count}: Fetched {len(result)} reviews so far.")
                time.sleep(1.5)

            for r in result[:max_reviews]:
                r['bank'] = bank
            all_reviews.extend(result[:max_reviews])

        except Exception as e:
            print(f"Error scraping {bank}: {e}. Using dummy reviews.")
            dummy = [{
                'content': f"Sample review {i} for {bank}",
                'score': np.random.randint(1, 6),
                'at': datetime.now(),
                'bank': bank
            } for i in range(max_reviews)]
            all_reviews.extend(dummy)

    return pd.DataFrame(all_reviews)


In [ ]:
def is_latin(text):
    """
    Checks if the majority of characters in a string are Latin.
    """
    text_alpha = ''.join([c for c in text if c.isalpha()])
    return len(text_alpha) / (len(text) + 1e-5) > 0.7

def is_english(text):
    """
    Uses langdetect to check if a string is English.
    """
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False


In [ ]:
def clean_reviews(df_raw):
    """
    Cleans the raw scraped DataFrame.
    Removes duplicates, nulls, non-English, and non-Latin reviews.
    """
    df = df_raw[['content', 'score', 'at', 'bank']].copy()
    df.columns = ['review', 'rating', 'date', 'bank']
    df['source'] = 'Google Play'

    print(f"Initial review count: {len(df)}")
    df['date'] = pd.to_datetime(df['date']).dt.date

    df.drop_duplicates(subset=['review', 'bank'], inplace=True)
    print(f"After removing duplicates: {len(df)}")

    df.dropna(subset=['review'], inplace=True)
    print(f"After dropping nulls: {len(df)}")

    df = df[df['review'].apply(is_latin)]
    print(f"After filtering non-Latin: {len(df)}")

    df = df[df['review'].apply(is_english)]
    print(f"After filtering non-English: {len(df)}")

    return df.reset_index(drop=True)


In [9]:
# Step 1: Scrape raw reviews
raw_df = scrape_reviews(apps, max_reviews=1000)

# Step 2: Clean and filter the reviews
df = clean_reviews(raw_df)

# Step 3: Save cleaned data to CSV
df.to_csv("bank_reviews_clean.csv", index=False)
print(f"Cleaned dataset saved. Total reviews: {len(df)}")



Scraping reviews for CBE...
Batch 1: Fetched 200 reviews.
Batch 2: Fetched 400 reviews so far.
Batch 3: Fetched 600 reviews so far.
Batch 4: Fetched 800 reviews so far.
Batch 5: Fetched 1000 reviews so far.

Scraping reviews for BOA...
Batch 1: Fetched 200 reviews.
Batch 2: Fetched 400 reviews so far.
Batch 3: Fetched 600 reviews so far.
Batch 4: Fetched 800 reviews so far.
Batch 5: Fetched 1000 reviews so far.

Scraping reviews for Dashen...
Batch 1: Fetched 200 reviews.
Batch 2: Fetched 400 reviews so far.
Batch 3: Fetched 448 reviews so far.
Batch 4: Fetched 448 reviews so far.
Batch 5: Fetched 448 reviews so far.
Batch 6: Fetched 448 reviews so far.
Batch 7: Fetched 448 reviews so far.
Batch 8: Fetched 448 reviews so far.
Batch 9: Fetched 448 reviews so far.
Batch 10: Fetched 448 reviews so far.
Initial review count: 2448
After removing duplicates: 2004
After dropping nulls: 2004
After filtering non-Latin: 1902
After filtering non-English: 1401
Cleaned dataset saved. Total reviews

In [10]:
# Load and display the CSV file
df_check = pd.read_csv("bank_reviews_clean.csv")
df_check.head()


,review,rating,date,bank,source
0,"""Why don’t your ATMs support account-to-accoun...",4,2025-06-06,CBE,Google Play
1,what is this app problem???,1,2025-06-05,CBE,Google Play
2,the app is proactive and a good connections.,5,2025-06-05,CBE,Google Play
3,I cannot send to cbebirr app. through this app.,3,2025-06-05,CBE,Google Play
4,not functional,1,2025-06-05,CBE,Google Play


In [11]:
from google.colab import files
files.download("bank_reviews_clean.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>